# C13_E01 — Gain scheduling para lazo de pH

**Capítulo:** 13 — Control adaptativo, autoajuste e identificación online  
**Identificador:** `C13_E01`  
**Objetivo pedagógico:** Demostrar la necesidad de gain scheduling en procesos no lineales fuertes.  
**Librerías:** numpy, matplotlib

## Ejemplo industrial

Lazo de pH en unidad de neutralización con tres regímenes operativos.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Tres regímenes de operación con PIDs distintos

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Ganancia del proceso depende del régimen (lazo de pH simplificado)
def K_proceso(pH):
    """Ganancia del proceso en función del régimen de pH."""
    if pH < 5: return 0.3      # ácido: pendiente suave
    elif pH > 9: return 0.3    # básico: pendiente suave
    else: return 5.0           # neutro: pendiente alta

# Tres PIDs sintonizados (Kc inversamente proporcional a K)
def Kc_para_region(pH):
    K = K_proceso(pH)
    return 0.5/K  # ≈ 1.67 en extremos, ≈ 0.1 en zona neutra

# Simulación: arranque desde pH=3 hasta SP=7
def simular(con_scheduling=True):
    Ts = 0.5; T = 100; t = np.arange(0, T, Ts)
    pH = 3.0; e_int = 0.0; pHs = []
    SP = 7.0; tau = 5.0
    for _ in t:
        e = SP - pH
        Kc = Kc_para_region(pH) if con_scheduling else 0.5/5.0  # PI fijo zona neutra
        u = Kc*e + 0.05*Kc*e_int
        e_int += e*Ts
        pH = pH + Ts*(K_proceso(pH)*u - pH)/tau
        pHs.append(pH)
    return t, np.array(pHs)

t, pH_gs = simular(True)
_, pH_no = simular(False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, pH_gs, label="Con gain scheduling")
ax.plot(t, pH_no, '--', label="PID lineal único")
ax.axhline(7.0, ls=':', color='gray', label="SP")
ax.set_xlabel("t (s)"); ax.set_ylabel("pH")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C13_E01 — pH: PID lineal vs gain scheduling")
out = '../../figures/cap13/fig_C13_F01_gain_scheduling.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 2. Interpretación

El PID lineal único, sintonizado para la zona neutra, es excesivamente agresivo en zona ácida o básica donde la ganancia del proceso es mucho menor (la curva de titración es plana). El gain scheduling adapta `Kc` al régimen actual, manteniendo desempeño consistente en los tres regímenes. Implementación industrial: tabla de PIDs con conmutación bumpless según variable de programación (en este caso, el propio pH).